### (Rerun if .pt model required) 
# Guwahati LSTM - Lucknow training method

This notebook adapts the Lucknow workflow: one multi-output PyTorch LSTM, 96-step lookback, one-step-ahead prediction, 70/15/15 chronological split, train-only feature and target scaling, early stopping, persistence baseline, R2, and range-specific metrics. Unlike the older Lucknow-filled data, unresolved Guwahati targets are masked and never used as labels.

In [2]:
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset, TensorDataset
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PROJECT_ROOT = Path.cwd().parent
DATA_FILE = PROJECT_ROOT / "08_unified_dataset" / "guwahati_unified_lstm_dataset_v2_bounded_cloud_fill_20240811_20260131.csv"
ARTIFACT_DIR = PROJECT_ROOT / "10_models" / "guwahati_lucknow_method"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COLUMNS = ["CLOUD_Vertical_Visibility", "VIS_Visibility_MOR"]
TARGET_IMPUTED_FLAGS = {"CLOUD_Vertical_Visibility": "CLOUD_Vertical_Visibility_IMPUTED"}
LOOKBACK = 96
HORIZON = 1
TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.15
BATCH_SIZE = 256
EPOCHS = 50
PATIENCE = 8

FEATURE_COLUMNS = [
    "CLOUD_Cloud_Base_1", "CLOUD_Cloud_Base_2", "CLOUD_Cloud_Base_3",
    "CLOUD_OCTA1", "CLOUD_OCTA2", "CLOUD_OCTA3", "CLOUD_Vertical_Visibility",
    "PTU_Pressure_QNH", "PTU_Temperature", "PTU_DewPoint", "PTU_Humidity",
    "PW_Rain_Intensity", "PW_Rain_Sum", "VIS_Visibility_MOR", "VIS_RVR",
    "WIND_Wind_Speed", "WIND_Wind_Gust", "Hour_Sin", "Hour_Cos",
    "CLOUD_SENSOR_ON", "PTU_SENSOR_ON", "PW_SENSOR_ON", "VIS_SENSOR_ON",
    "WIND_SENSOR_ON", "CLOUD_Cloud_Base_1_IMPUTED", "CLOUD_Cloud_Base_2_IMPUTED",
    "CLOUD_Cloud_Base_3_IMPUTED", "CLOUD_Vertical_Visibility_IMPUTED",
    "CLOUD_LONG_GAP", "CLOUD_VERTICAL_LONG_GAP", "WIND_Wind_Dir_Sin", "WIND_Wind_Dir_Cos",
]
assert DATA_FILE.exists(), DATA_FILE
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)


PyTorch: 2.5.1
Device: cuda


In [3]:
df = pd.read_csv(DATA_FILE, parse_dates=["Datetime"], dayfirst=True)
df = df.sort_values("Datetime").drop_duplicates("Datetime")
raw_required = sorted((set(FEATURE_COLUMNS + TARGET_COLUMNS) - {"WIND_Wind_Dir_Sin", "WIND_Wind_Dir_Cos"}) - set(df.columns))
assert not raw_required, f"Missing raw columns: {raw_required}"
numeric = sorted((set(FEATURE_COLUMNS + TARGET_COLUMNS) - {"WIND_Wind_Dir_Sin", "WIND_Wind_Dir_Cos"}) & set(df.columns))
df[numeric] = df[numeric].apply(pd.to_numeric, errors="coerce")
df.loc[df["WIND_Wind_Speed"] < 0, "WIND_Wind_Speed"] = np.nan
df.loc[df["WIND_Wind_Gust"] < 0, "WIND_Wind_Gust"] = np.nan
df["WIND_Wind_Dir_Sin"] = np.sin(np.deg2rad(df["WIND_Wind_Dir"]))
df["WIND_Wind_Dir_Cos"] = np.cos(np.deg2rad(df["WIND_Wind_Dir"]))
df = df.drop(columns=["WIND_Wind_Dir"]).set_index("Datetime")
df["ROW_GAP"] = df.index.to_series().diff().dt.total_seconds().gt(15).fillna(False)
print("Rows:", len(df), "range:", df.index.min(), df.index.max())


Rows: 2848084 range: 2024-08-11 00:00:00 2026-01-31 10:08:45


In [4]:
feature_scaler = StandardScaler()
target_scaler = StandardScaler()
train_end = int(len(df) * TRAIN_RATIO)
validation_end = int(len(df) * (TRAIN_RATIO + VALIDATION_RATIO))
feature_medians = df.iloc[:train_end][FEATURE_COLUMNS].median()
feature_scaler.fit(df.iloc[:train_end][FEATURE_COLUMNS].fillna(feature_medians))
prepared = pd.DataFrame(feature_scaler.transform(df[FEATURE_COLUMNS].fillna(feature_medians)), index=df.index, columns=FEATURE_COLUMNS).fillna(0).astype("float32")
target_training = df.iloc[:train_end][TARGET_COLUMNS].copy()
for target, flag in TARGET_IMPUTED_FLAGS.items():
    target_training.loc[df.iloc[:train_end][flag].fillna(0).eq(1), target] = np.nan
target_scaler.fit(target_training.dropna(how="all").fillna(target_training.median()))
target_medians = target_training.median()
with open(ARTIFACT_DIR / "scalers.json", "w", encoding="utf-8") as handle:
    json.dump({"features": FEATURE_COLUMNS, "feature_medians": feature_medians.to_dict(), "target_means": dict(zip(TARGET_COLUMNS, target_scaler.mean_)), "target_scales": dict(zip(TARGET_COLUMNS, target_scaler.scale_))}, handle, indent=2)

def observed_targets(frame):
    mask = frame[TARGET_COLUMNS].notna().to_numpy()
    for index, target in enumerate(TARGET_COLUMNS):
        flag = TARGET_IMPUTED_FLAGS.get(target)
        if flag:
            mask[:, index] &= frame[flag].fillna(0).eq(0).to_numpy()
    return mask

values = prepared.to_numpy(dtype="float32")
targets = df[TARGET_COLUMNS].to_numpy(dtype="float32")
masks = observed_targets(df)
gaps = df["ROW_GAP"].to_numpy(dtype=bool)
window_ends, labels, label_masks, timestamps = [], [], [], []
for end in range(LOOKBACK, len(df) - HORIZON + 1):
    horizon = end + HORIZON - 1
    if gaps[end - LOOKBACK + 1:horizon + 1].any() or not masks[horizon].any():
        continue
    label = targets[horizon].copy()
    label_mask = masks[horizon].copy()
    label[~label_mask] = target_medians.to_numpy()[~label_mask]
    window_ends.append(end)
    labels.append(label)
    label_masks.append(label_mask.astype("float32"))
    timestamps.append(df.index[horizon])
window_ends = np.asarray(window_ends, dtype="int64")
y_original = np.asarray(labels, dtype="float32")
mask = np.asarray(label_masks, dtype="float32")
timestamps = pd.to_datetime(timestamps)
y_scaled = target_scaler.transform(y_original).astype("float32")
train = timestamps < timestamps[int(len(timestamps) * TRAIN_RATIO)]
validation = (timestamps >= timestamps[int(len(timestamps) * TRAIN_RATIO)]) & (timestamps < timestamps[int(len(timestamps) * (TRAIN_RATIO + VALIDATION_RATIO))])
test = timestamps >= timestamps[int(len(timestamps) * (TRAIN_RATIO + VALIDATION_RATIO))]
print("Windows:", len(window_ends), "train/validation/test:", int(train.sum()), int(validation.sum()), int(test.sum()))


Windows: 1885202 train/validation/test: 1319641 282780 282781


c:\Users\iiiaamnooob\miniconda3\envs\weather_pipe\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [5]:
class MultiOutputLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(len(FEATURE_COLUMNS), 64, batch_first=True)
        self.head = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, len(TARGET_COLUMNS)))
    def forward(self, values):
        output, _ = self.lstm(values)
        return self.head(output[:, -1, :])

class WindowDataset(torch.utils.data.Dataset):
    def __len__(self):
        return len(window_ends)
    def __getitem__(self, index):
        end = window_ends[index]
        return (torch.from_numpy(values[end - LOOKBACK:end]),
                torch.from_numpy(y_scaled[index]),
                torch.from_numpy(mask[index]))

dataset = WindowDataset()

def loader(split_mask):
    indices = np.flatnonzero(split_mask).tolist()
    return DataLoader(Subset(dataset, indices), batch_size=BATCH_SIZE, shuffle=False)

train_loader = loader(train)
validation_loader = loader(validation)
model = MultiOutputLSTM().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss(reduction="none")
best_loss, best_state, no_improvement = float("inf"), None, 0
for epoch in range(EPOCHS):
    model.train()
    for inputs, labels, label_mask in train_loader:
        inputs, labels, label_mask = inputs.to(DEVICE), labels.to(DEVICE), label_mask.to(DEVICE)
        optimizer.zero_grad()
        loss = (criterion(model(inputs), labels) * label_mask).sum() / label_mask.sum().clamp_min(1)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
    model.eval()
    losses = []
    with torch.no_grad():
        for inputs, labels, label_mask in validation_loader:
            losses.append(((criterion(model(inputs.to(DEVICE)), labels.to(DEVICE)) * label_mask.to(DEVICE)).sum() / label_mask.sum().clamp_min(1)).item())
    validation_loss = float(np.mean(losses))
    print(f"Epoch {epoch + 1:03d}/{EPOCHS} - val_loss: {validation_loss:.6f}")
    if validation_loss < best_loss:
        best_loss, no_improvement = validation_loss, 0
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
    else:
        no_improvement += 1
        if no_improvement >= PATIENCE:
            break
model.load_state_dict(best_state)
torch.save({"model_state_dict": model.state_dict(), "features": FEATURE_COLUMNS, "targets": TARGET_COLUMNS}, ARTIFACT_DIR / "best_guwahati_lucknow_method.pt")


Epoch 001/50 - val_loss: 0.112229
Epoch 002/50 - val_loss: 0.150764
Epoch 003/50 - val_loss: 0.049706
Epoch 004/50 - val_loss: 0.069175
Epoch 005/50 - val_loss: 0.150125
Epoch 006/50 - val_loss: 0.049570
Epoch 007/50 - val_loss: 0.047501
Epoch 008/50 - val_loss: 0.051743
Epoch 009/50 - val_loss: 0.054782
Epoch 010/50 - val_loss: 0.063356
Epoch 011/50 - val_loss: 0.034807
Epoch 012/50 - val_loss: 0.043282
Epoch 013/50 - val_loss: 0.054091
Epoch 014/50 - val_loss: 0.048774
Epoch 015/50 - val_loss: 0.028973
Epoch 016/50 - val_loss: 0.037430
Epoch 017/50 - val_loss: 0.040248
Epoch 018/50 - val_loss: 0.043294
Epoch 019/50 - val_loss: 0.038045
Epoch 020/50 - val_loss: 0.045899
Epoch 021/50 - val_loss: 0.037941
Epoch 022/50 - val_loss: 0.028774
Epoch 023/50 - val_loss: 0.034522
Epoch 024/50 - val_loss: 0.043718
Epoch 025/50 - val_loss: 0.042925
Epoch 026/50 - val_loss: 0.033119
Epoch 027/50 - val_loss: 0.032214
Epoch 028/50 - val_loss: 0.042182
Epoch 029/50 - val_loss: 0.031690
Epoch 030/50 -

In [6]:
test_loader = loader(test)
model.eval()
prediction_batches = []
with torch.no_grad():
    for inputs, _, _ in test_loader:
        prediction_batches.append(model(inputs.to(DEVICE)).cpu().numpy())
predictions_scaled = np.concatenate(prediction_batches, axis=0)
predictions = target_scaler.inverse_transform(predictions_scaled)
actual = y_original[test]
valid = mask[test].astype(bool)
rows = []
for index, target in enumerate(TARGET_COLUMNS):
    actual_target, predicted_target = actual[valid[:, index], index], predictions[valid[:, index], index]
    rows.append({"target": target, "samples": len(actual_target), "MAE": mean_absolute_error(actual_target, predicted_target), "RMSE": mean_squared_error(actual_target, predicted_target) ** 0.5, "R2": r2_score(actual_target, predicted_target)})
metrics = pd.DataFrame(rows)
metrics.to_csv(ARTIFACT_DIR / "test_metrics.csv", index=False)
display(metrics)

range_rows = []
for index, target in enumerate(TARGET_COLUMNS):
    actual_target, predicted_target = actual[valid[:, index], index], predictions[valid[:, index], index]
    for name, range_mask in [("low", actual_target < 1000), ("medium", (actual_target >= 1000) & (actual_target < 5000)), ("high", actual_target >= 5000)]:
        if range_mask.sum() > 1:
            range_rows.append({"target": target, "range": name, "samples": int(range_mask.sum()), "MAE": mean_absolute_error(actual_target[range_mask], predicted_target[range_mask]), "RMSE": mean_squared_error(actual_target[range_mask], predicted_target[range_mask]) ** 0.5, "R2": r2_score(actual_target[range_mask], predicted_target[range_mask])})
pd.DataFrame(range_rows).to_csv(ARTIFACT_DIR / "range_test_metrics.csv", index=False)


,target,samples,MAE,RMSE,R2
0,CLOUD_Vertical_Visibility,4348,61.083427,146.568057,0.974409
1,VIS_Visibility_MOR,282370,138.589676,199.653665,0.994723
